# VisTacFusion Co-Training (Colab)

Train VisTacFusion (sim + real co-training) on Google Colab.

**Steps:**
1. Setup environment (clone repo, install deps)
2. Download / cache pretrained encoders (T3 + MAE) on Google Drive
3. Mount data
4. Configure and launch training
5. Monitor, evaluate, save checkpoints to Drive

## 1. Environment Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive (for caching encoders + saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo
!git clone https://github.com/cynthiahuang1004/VisTacFusion.git
%cd VisTacFusion
!git checkout VisTacFusion-v2

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers safetensors tensorboard opencv-python matplotlib pyyaml tqdm timm

import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')

## 2. Download Pretrained Encoders (cached on Drive)

First run downloads T3 + MAE to Google Drive. Subsequent runs just symlink from Drive.

In [ ]:
import os, shutil

DRIVE_ENCODER_DIR = '/content/drive/MyDrive/VisTacFusion_encoders'
LOCAL_ENCODER_DIR = 'pretrained_encoders'

os.makedirs(DRIVE_ENCODER_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_ENCODER_DIR}/t3_large', exist_ok=True)
os.makedirs(f'{LOCAL_ENCODER_DIR}/t3_large', exist_ok=True)

ENCODER_FILES = {
    # T3 Large (tactile encoder)
    't3_large/encoder_mini.pth': 'https://huggingface.co/datasets/alanz-mit/FoundationTactile/resolve/main/models/t3_large/encoders/mini.pth',
    't3_large/trunk.pth': 'https://huggingface.co/datasets/alanz-mit/FoundationTactile/resolve/main/models/t3_large/trunk.pth',
    # MAE ViT-L/16 (RGB encoder)
    'mae_vitl16.pth': 'https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_large.pth',
}

for local_name, url in ENCODER_FILES.items():
    drive_path = f'{DRIVE_ENCODER_DIR}/{local_name}'
    local_path = f'{LOCAL_ENCODER_DIR}/{local_name}'
    
    if os.path.exists(drive_path):
        size_mb = os.path.getsize(drive_path) / 1e6
        print(f'[cached] {local_name} ({size_mb:.0f} MB) <- Drive')
    else:
        print(f'[downloading] {local_name} ...')
        !wget -q --show-progress -O "{drive_path}" "{url}"
        size_mb = os.path.getsize(drive_path) / 1e6
        print(f'  saved to Drive ({size_mb:.0f} MB)')
    
    # Symlink from Drive to local
    if os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)

print('\nLocal encoder dir:')
!ls -lh {LOCAL_ENCODER_DIR}/
!ls -lh {LOCAL_ENCODER_DIR}/t3_large/

## 3. Mount Data

Set the paths to your sim and real datasets. Expected structure:

```
<data_root>/
  <object_name>/
    <session_name>/
      sensor_0/
        <idx>.png          # tactile
        <idx>_gt.npy       # depth GT
        <idx>_pose.json    # pose GT
        rgb/<idx>.png      # RGB
        norms/             # (optional) rendered normals
      session.json
```

In [ ]:
# ===================== SET YOUR DATA PATHS =====================

SIM_ROOT = ''           # e.g. '/content/drive/MyDrive/renders_v3'
REAL_ROOT = ''          # e.g. '/content/drive/MyDrive/real_data'
MESH_DIR = ''           # e.g. '/content/drive/MyDrive/meshes'

# ==============================================================

for name, path in [('SIM_ROOT', SIM_ROOT), ('REAL_ROOT', REAL_ROOT), ('MESH_DIR', MESH_DIR)]:
    if path:
        exists = os.path.isdir(path)
        n = len(os.listdir(path)) if exists else 0
        status = f'OK, {n} items' if exists else 'NOT FOUND'
        print(f'{name}: {path} ({status})')
    else:
        print(f'{name}: (not set)')

## 4. Configure Training

In [ ]:
import yaml

# ===================== TRAINING SETTINGS =====================

MODE = 'sim+real'                    # 'sim' or 'sim+real'
SIM_SAMPLES_PER_SESSION = None       # None = all, int = subsample
REAL_VAL_EVERY = 10
SIM_VAL_EVERY = 20

MODEL_CONFIG = 'ablation/encoder/tac_t3_rgb_mae.yaml'
TRAIN_CONFIG = 'configs/train_bs32.yaml'
OUTPUT_DIR = 'outputs/colab_co_train'
RESUME_CKPT = None                   # or f'{OUTPUT_DIR}/latest.pt'

# ==============================================================

data_cfg = {
    'image_size': 224,
    'dataset': MODE,
    'synthetic': {'num_samples': 256, 'num_objects': 8},
    'sim': {
        'root': SIM_ROOT,
        'mesh_dir': MESH_DIR,
        'rgb_subdir': 'rgb',
        'use_gt_depth': True,
        'use_rendered_normals': True,
        'gel_view_m': 0.017502,
        'rot_augment': True,
        'rot_augment_max_deg': 180.0,
        'val_every': SIM_VAL_EVERY,
    },
    'real': {
        'root': REAL_ROOT,
        'mesh_dir': MESH_DIR,
        'rgb_subdir': 'rgb',
        'use_rendered_normals': False,
        'val_every': REAL_VAL_EVERY,
        'augment': False,
        'oversample': 1,
    },
    'loader': {
        'num_workers': 4,
        'pin_memory': True,
        'prefetch_factor': 4,
        'persistent_workers': True,
    },
    'norm': {
        'imagenet_mean': [123.675, 116.28, 103.53],
        'imagenet_std': [58.395, 57.12, 57.375],
    },
}

if SIM_SAMPLES_PER_SESSION is not None:
    data_cfg['sim']['train_samples_per_session'] = SIM_SAMPLES_PER_SESSION

data_config_path = 'configs/data_colab.yaml'
with open(data_config_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Model:  {MODEL_CONFIG}')
print(f'Train:  {TRAIN_CONFIG}')
print(f'Data:   {data_config_path}')
print(f'Output: {OUTPUT_DIR}')
print(f'Resume: {RESUME_CKPT}')
print(f'\n--- Data Config ---')
print(yaml.dump(data_cfg, default_flow_style=False))

## 5. Verify Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from vistacfusion.engine.train import merge_configs
from vistacfusion.models.model import build_model
from vistacfusion.data.dataset import build_datasets

cfg = merge_configs(MODEL_CONFIG, TRAIN_CONFIG, data_config_path)

train_ds, val_ds = build_datasets(cfg)
print(f'Train: {len(train_ds)} samples')
print(f'Val:   {len(val_ds)} samples')

model = build_model(cfg)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {type(model).__name__} ({total/1e6:.1f}M total, {trainable/1e6:.1f}M trainable)')

# Quick forward test
model.eval()
with torch.no_grad():
    x = torch.randn(2, 3, 224, 224)
    out = model(x, x, config='both', object_ids=torch.zeros(2, dtype=torch.long))
    print(f'Forward OK: depth={out["depth"].shape}, se2={out["se2"].shape}')

del model, train_ds, val_ds
torch.cuda.empty_cache()
print('Setup verified!')

## 6. Train

In [ ]:
resume_arg = f'--resume {RESUME_CKPT}' if RESUME_CKPT else ''

!python -u -m vistacfusion.engine.train \
  --model {MODEL_CONFIG} \
  --train {TRAIN_CONFIG} \
  --data {data_config_path} \
  --output-dir {OUTPUT_DIR} \
  {resume_arg}

## 7. Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

history_path = f'{OUTPUT_DIR}/history.json'
if not os.path.exists(history_path):
    print(f'{history_path} not found.')
else:
    with open(history_path) as f:
        history = json.load(f)
    
    epochs = [h['epoch'] for h in history]
    cfg_key = 'both' if 'both' in history[0]['val'] else 'tactile'
    
    metrics = [
        ('depth_mse', 'Depth MSE', '#2a78d6'),
        ('normal_mse', 'Normal MSE', '#1baf7a'),
        ('pose_rot_deg', 'Rotation (deg)', '#eb6834'),
        ('pose_rot_l1', 'Rotation L1', '#e87ba4'),
        ('pose_trans', 'Translation L1', '#4a3aa7'),
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'{os.path.basename(OUTPUT_DIR)} (epoch {epochs[-1]})', fontsize=14, fontweight='bold')
    
    for ax, (key, label, color) in zip(axes.flat, metrics):
        vals = [h['val'][cfg_key].get(key) for h in history]
        vals = [v for v in vals if v is not None]
        if vals:
            ax.plot(epochs[:len(vals)], vals, color=color, lw=1.8)
            best_i = int(np.argmin(vals))
            ax.scatter(epochs[best_i], vals[best_i], color=color, s=60, zorder=5)
            ax.set_title(f'{label} (best: {vals[best_i]:.4f} @ e{epochs[best_i]})', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    
    axes.flat[-1].axis('off')
    plt.tight_layout()
    plt.show()
    
    last = history[-1]['val'][cfg_key]
    print(f"Final (epoch {epochs[-1]}):")
    for key, label, _ in metrics:
        if key in last:
            print(f'  {label:20s} {last[key]:.6f}')

## 8. Save Checkpoints to Drive

In [ ]:
DRIVE_SAVE_DIR = f'/content/drive/MyDrive/VisTacFusion_checkpoints/{os.path.basename(OUTPUT_DIR)}'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for name in ['best_depth.pt', 'best_pose.pt', 'latest.pt', 'history.json']:
    src = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(src):
        dst = os.path.join(DRIVE_SAVE_DIR, name)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  {name} ({size_mb:.0f} MB)')
    else:
        print(f'  {name} (not found)')

print(f'\nSaved to {DRIVE_SAVE_DIR}/')